# CFPB Privacy NER QA

Default scope is the 20-row engineering smoke input. Change `SCAN_SCOPE` to `full_v052` only after Seed v05.2 exists. This uses Presidio with spaCy `en_core_web_sm`; every positive requires manual review.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, json, subprocess, sys
import pandas as pd

SCAN_SCOPE = "smoke20"  # smoke20 or full_v052
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/FinDisputeEval")
else:
    here = Path.cwd().resolve()
    ROOT = next((p for p in (here, *here.parents) if (p / "WORK_PROGRESS.md").exists()), None)
    if ROOT is None: raise FileNotFoundError("Open the repository")


In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas>=2.2,<3", "pyarrow>=16", "spacy>=3.8,<4", "presidio-analyzer>=2.2,<3"])
check = subprocess.run([sys.executable, "-c", "import spacy; spacy.load('en_core_web_sm')"], check=False)
if check.returncode != 0:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])


In [ ]:
RUNTIME = Path("/content/findisputeeval_privacy") if IN_COLAB else ROOT / ".runtime/privacy_qa"
PACKAGE = RUNTIME / "findisputeeval/curation"
PACKAGE.mkdir(parents=True, exist_ok=True)
(RUNTIME / "findisputeeval/__init__.py").write_text("", encoding="utf-8")
(PACKAGE / "__init__.py").write_text("", encoding="utf-8")
raw = gzip.decompress(base64.b64decode("H4sIAAAAAAAC/51YbW/bNhD+7l9B8MMgbYqWdu0wGPOALPFWY6lj2GmBwTAIxqITbnobKSVxXfe3744v1ovjbVjQ2iJ5d7x77uHxZErpTAktE1l8q0t+uSWlko98vSV6zXPC84RsuEzP1mmhRUIyntc8JUo8SvFEHkRaCqVjSulgsFFFRhjb1FWtBGNEZmWhKrCQFxWvZJHrwcDNPXD9kMo7P/xDF7lVT3glKpkJr+zHEcHPT0UurFzJKzTgxWYwtAvVtpT5vZ+/yLeHPUsIhWsC/8pkMBi8m/z6js0ni9/YeHo7uZ2MF2REdgMCf3Q2ni9upjSyo+uby4vbSTOevbuZjtn0w/ufx3M/N35/MblmF1dX8/Fi4Sc/LNhicVC7nI+vJrfs8mJ+1RL4+WL6W88WzF7NJx/Hc3Y9uRxPF+PWwuxisZjdzG/91GTW33QCFtnlzdW42ff32e2NH70HJy4vrvump/MZPO4Bl0RsiH7gr99+zzYyFQEiPTQAh+TsJ6IrNTQqibwXugLMXC5jqxSEZvVJVg9GyeiHcVGKPKDqjoaYgAdIRSqsHfzbFIrcpcX6TyJzIiuhgpRndwkfOslYCZ4Er85fvyFfE/wKI3JHadhYaDyK6xJJExh71hklgJG5X38Qz/YJXLXh3tUyTVjpTgHjOU+3n8CJrEhEynKeiSHGDbFSkbN1Aex+EndMZ94DS8m+fkNCOx7n9xL4e1ohztOSCSPldadpadVmqniUiVADo1+6Ebh0JBEcQEFz6yLfyPtamQM42nUAo81+Jko6JBRKwHrrWHGQM0BoWF7uaMrze7CaGGmR08gtewvNYL9qzOzNUycdXViCxpeRjy5eQ94r4aYDSLquS4RFJAzdqDmkcbREL1Y+l1i02EbB9oFD2qSvTOIrXvFfcGS98qAPsUjYqa/tl0wgvrTOcpN1O1eJ5+p4Vpep7EyTz2QKNQrTAl9WKJO5zOqMaSTOkGzSguOxOY+/exsNzJlqOzd0OlpjGRsRbcINdgenorYve3JGtKgCE2Vs53RoYZYbb6Y5JopLLchHntZirFShgg0U/1a1dxYIV+KgTHbuaU9d/oonPSSp1NUyketqiXAgiKsV+LtcDfyJVgIiTvBIW/eqgqF8UCgp8mpE7bpuH2MMDYOuVGBX43uIrhUwsI26+Pw2G5knpuTnh5zG7sFojvAjIp4wI0ta8EBWEujjAD6+DsJeeQE8Te4Ct2FsEhqSH3sZ7ijhHxzBSua16CzoiqsK3UggXpm3rOICUL09B1Jht9SJSqzBbdDF6JZGaQhiq46Y02cyOa7TR25u6M5ivjxwbbX/vGt8AMC2DO5XAbNmQ/iGLeHTu7OnIIa1IaB1tTn7gXa9Dtu1dzl8/abrLfIq5iXcFMmxd7ujGVOZmgih9DSD6GVpG58VbjjWijc8qZgKrgUzB76jbAjargOWociW9iyB+ilg5YT9FrjO+guwn3LO0I4OX2TnKRVMn9kJWXjKJ4QJPk+s+6QzZCBI+vEJcTwFVtAwNuPPwXlkHYAi9ur8PCRDPElBCp0CioT2dHxj1lYnrNp2lJVSmvsb0MJrCSkEGNB/ySZf46X4nxSw2xUKRf9FJmF1tf4nOWiGhT4hsO8el4GtHrYkQ2kdvET7qE3rqE/VqMutyLMl8hyIbKIbX3p5jZrMRS/CHR3hGbUAa9ntABR5IKzAqt0YtK/DAItC5EEY+RsuxqrNHvEe002tWHaRaAXYILCKyJ8A3ohmQkEhAjPUtiYxxlMxWBPPQaKKcnSrauHbCgCcp/KTYO7liNloAuc3Pr/UZLiOAjAvRbtvKGq1hhSZSuzmTSfQvU/tVcLTtEDYHOD4lkK3iBxCSC1lUn4HBQaWrCvLlxK1irlGCAKwDwBWKka7yjXsMgc0zT1hTcDievnF2o0lXP9Bz49w1e5UmM9414B358CHYyfgvyyDMBZ/BVA4V751AXZ4n2KRldWWwFWPk/0N7eo/tDjdG/HQ77g3WAn3VVamwPmh348VirmCMNphNXLz4R4g71nre2MV+rPhvtGzcJeFhvbjUeguXg5vxAJT7ODwwu6Aoc5Bf9k/ff8pzS9Zshnv72Vzv6PieZ3WibCHO4FFeGtJIEBsG+neOQoUaVlukv8/HMzFPUc/TuID3A/JV+RLa0/vq2HqwakumQ6yHVL1d/2/pPLBO3vDA7odInkfkFB+x66Anz0mjquRTTtE14Cu4jnUk0f4KcZdaI/nr9q11xQgvO/xO+rcI6as+VvL/9wS58VT4H9xiWEtBGwL6LYz6DDCtuF2JcMN2uPo6MLS/khga4EB22HbonlhVRkWGiheXs+JH8gadi4Wy68GiJJrbbZoyG0S2lH6q5bAZLYpkC2sLkG8m9blCrnTM2EbuSWFPjXdesTBTAGPCl5aNVYPe0IOd6H9faF1Xpq+18WxH/wNq0+NtYcTAAA="))
if hashlib.sha256(raw).hexdigest() != "a9654f9bf4453bb4dec29ad7b86f384f9c3005b2649a8d03d2a3286185dc1965": raise ValueError("Embedded privacy module hash mismatch")
(PACKAGE / "cfpb_privacy_qa.py").write_bytes(raw)
sys.path.insert(0, str(RUNTIME))
from findisputeeval.curation.cfpb_privacy_qa import build_presidio_analyzer, finalize_privacy_review, scan_frame, sha256_file


In [ ]:
if SCAN_SCOPE == "smoke20":
    source = ROOT / "outputs/generation/smoke_only/cfpb_seed_v051_candidate/prepared_inputs/nemo_seed_v051_provisional_smoke_20.jsonl"
    frame = pd.read_json(source, lines=True)
    id_column, text_column, split_column = "seed_id", "seed_narrative_excerpt", "release_split"
elif SCAN_SCOPE == "full_v052":
    release = ROOT / "dataset/curated/seed_pools/cfpb_dispute/seed_v052"
    seed = pd.read_parquet(release / "cfpb_seed_v052.parquet")
    stress = pd.read_parquet(release / "cfpb_seed_v052_stress_test.parquet").rename(columns={"stress_id": "seed_id"})
    frame = pd.concat([seed, stress], ignore_index=True)
    source = release / "seed_v052_manifest.json"
    id_column, text_column, split_column = "seed_id", "seed_text", "release_split"
else:
    raise ValueError(SCAN_SCOPE)
if not source.exists(): raise FileNotFoundError(source)
QA_ROOT = ROOT / "dataset/curated/annotations/cfpb_seed_v05_audit/run_20260713T145423Z/privacy_qa" / SCAN_SCOPE
QA_ROOT.mkdir(parents=True, exist_ok=True)
REVIEW = QA_ROOT / "presidio_spacy_review.csv"
CLEARANCE = QA_ROOT / "privacy_clearance.json"
print({"scope": SCAN_SCOPE, "rows": len(frame), "source": str(source), "qa_root": str(QA_ROOT)})


In [ ]:
# Scan once. Existing manual review is never overwritten.
if REVIEW.exists():
    review = pd.read_csv(REVIEW, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    print("Existing review loaded:", len(review))
else:
    analyzer = build_presidio_analyzer("en_core_web_sm")
    review = scan_frame(frame, analyzer, id_column=id_column, text_column=text_column, split_column=split_column, minimum_score=0.35)
    review.to_csv(REVIEW, index=False, encoding="utf-8-sig")
    print("NER findings:", len(review))
display(review.head(50))


In [ ]:
# Interactive manual review; progress is saved after each answer.
for index in review.index[review.manual_pii_present.eq("pending")]:
    row = review.loc[index]
    print("\n", row.record_id, row.entity_type, row.score)
    print(row.context)
    label = input("Actual PII? [yes/no/stop]: ").strip().lower()
    if label == "stop": break
    if label not in {"yes", "no"}: print("Invalid; unchanged"); continue
    review.at[index, "manual_pii_present"] = label
    review.at[index, "release_action"] = "allow" if label == "no" else input("Action [exclude/redact_and_rescan]: ").strip().lower()
    review.at[index, "reviewer"] = "chang"
    review.at[index, "reviewed_utc"] = pd.Timestamp.utcnow().isoformat()
    review.at[index, "notes"] = input("Optional note: ").strip()
    review.to_csv(REVIEW, index=False, encoding="utf-8-sig")
print("Pending:", review.manual_pii_present.eq("pending").sum())


In [ ]:
result = finalize_privacy_review(review, scope=SCAN_SCOPE, source_sha256=sha256_file(source))
CLEARANCE.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, indent=2))
if not result["release_clearance_passed"]:
    print("BLOCKED: apply versioned exclusions/redaction, rebuild, then rescan.")
